# ProxyLens 03: export a trained checkpoint (no training)

**When:** notebook 01 finished training but the session ended before the upload/GGUF cells (a
long Colab session can run out of time or GPU memory). This loads the latest checkpoint from
Google Drive in a fresh session and does only the remaining steps.

**Inputs:** the Drive folder notebook 01 wrote (`MyDrive/proxylens/qwen3b-lora-v1/checkpoint-*`);
Colab secret `HF_TOKEN`.

**Outputs:** the same as notebook 01: LoRA adapter + `metrics.json` at
`<you>/proxylens-qwen3b-lora-v1` and `proxylens-q4_k_m.gguf` at `<you>/proxylens-qwen3b-gguf-v1`.

**Runtime:** free T4, about 20-40 min (mostly the GGUF conversion).

In [ ]:
%%capture
# Unsloth pulls matching torch / transformers / trl / peft / bitsandbytes for Colab.
!pip install -q unsloth
!pip install -q "huggingface_hub>=0.25" "pydantic>=2.8"

In [ ]:
import os
# Must be set before torch starts: reduces fragmentation-driven OOMs on the 15 GB T4.
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

from unsloth import FastLanguageModel  # import before transformers/trl: Unsloth patches them

import glob, json, time
import torch
from google.colab import drive, userdata
from huggingface_hub import HfApi, snapshot_download

print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0))

HF_TOKEN = userdata.get("HF_TOKEN")  # Colab secret; needs write access
HF_USER = HfApi(token=HF_TOKEN).whoami()["name"]
DATASET_REPO = f"{HF_USER}/proxylens-sft-v1"           # pushed by scripts/push_sft_to_hub.py
LORA_REPO = f"{HF_USER}/proxylens-qwen3b-lora-v1"
GGUF_REPO = f"{HF_USER}/proxylens-qwen3b-gguf-v1"
GGUF_FILE = "proxylens-q4_k_m.gguf"                     # = STUDENT_GGUF_FILE in .env
BASE_MODEL = "unsloth/Qwen2.5-3B-Instruct-bnb-4bit"
MAX_SEQ_LEN = 4096        # longest SFT example is ~4.1k tokens
MAX_NEW_TOKENS = 1024     # longest teacher answer is ~870 tokens
SEED = 3407
print("HF user:", HF_USER)

import transformers
transformers.utils.logging.set_verbosity_error()  # hides a harmless per-generate max_length warning

In [ ]:
drive.mount("/content/drive")
OUTPUT_DIR = "/content/drive/MyDrive/proxylens/qwen3b-lora-v1"
CHECKPOINT = None   # e.g. f"{OUTPUT_DIR}/checkpoint-130" to choose one; None = latest complete

def complete(path):
    # Finished syncing to Drive: adapter weights + trainer state present.
    weights = os.path.join(path, "adapter_model.safetensors")
    return (os.path.isfile(weights) and os.path.getsize(weights) > 50_000_000
            and os.path.isfile(os.path.join(path, "trainer_state.json")))

found = sorted(glob.glob(f"{OUTPUT_DIR}/checkpoint-*"), key=lambda p: int(p.rsplit("-", 1)[1]))
for path in found:
    print(f"{os.path.basename(path):16} {'complete' if complete(path) else 'INCOMPLETE (skipped)'}")
if CHECKPOINT is None:
    usable = [p for p in found if complete(p)]
    assert usable, f"no complete checkpoint in {OUTPUT_DIR}"
    CHECKPOINT = usable[-1]
print("using", CHECKPOINT)

In [ ]:
# The checkpoint holds the LoRA adapter; Unsloth loads the 4-bit base model under it.
model, tokenizer = FastLanguageModel.from_pretrained(
    CHECKPOINT, max_seq_length=MAX_SEQ_LEN, load_in_4bit=True, dtype=None
)
FastLanguageModel.for_inference(model)

In [ ]:
# Copied from app/schemas/resolution.py by scripts/build_notebooks.py. Do not edit here.
"""Resolution extraction schema (SPEC 5.1). Also the fine-tuning target format."""

from enum import StrEnum

from pydantic import BaseModel, Field


class ResolutionType(StrEnum):
    ADOPT_FINANCIALS = "ADOPT_FINANCIALS"
    DIVIDEND = "DIVIDEND"
    DIRECTOR_REAPPOINT_ROTATION = "DIRECTOR_REAPPOINT_ROTATION"
    INDEPENDENT_DIRECTOR_APPOINT = "INDEPENDENT_DIRECTOR_APPOINT"
    NON_INDEPENDENT_DIRECTOR_APPOINT = "NON_INDEPENDENT_DIRECTOR_APPOINT"
    MANAGERIAL_REMUNERATION = "MANAGERIAL_REMUNERATION"
    RELATED_PARTY_TRANSACTION = "RELATED_PARTY_TRANSACTION"
    AUDITOR_APPOINT = "AUDITOR_APPOINT"
    AUDITOR_REMUNERATION_COST = "AUDITOR_REMUNERATION_COST"  # cost auditor ratification
    ESOP = "ESOP"
    CAPITAL_RAISE = "CAPITAL_RAISE"  # QIP, preferential, NCDs
    BORROWING_LIMITS = "BORROWING_LIMITS"  # Sec 180
    ARTICLES_OR_MOA_AMENDMENT = "ARTICLES_OR_MOA_AMENDMENT"
    OTHER = "OTHER"


class Person(BaseModel):
    name: str
    role: str | None = None
    age: int | None = None
    din: str | None = None
    is_promoter: bool | None = None
    tenure_years_proposed: float | None = None
    prior_tenure_years: float | None = None
    board_attendance_pct: float | None = None


class Money(BaseModel):
    amount_inr: float | None = None  # normalized to INR (crore/lakh -> INR)
    raw: str


class ResolutionExtraction(BaseModel):
    item_no: int
    title: str
    resolution_type: ResolutionType
    is_special_resolution: bool
    persons: list[Person] = Field(default_factory=list)
    amounts: list[Money] = Field(default_factory=list)
    counterparty: str | None = None
    counterparty_is_related: bool | None = None
    transaction_nature: str | None = None
    duration_years: float | None = None
    key_facts: list[str] = Field(default_factory=list, max_length=5)

In [ ]:
def generate(model, messages, max_new_tokens=MAX_NEW_TOKENS):
    """Greedy decode for one chat (system + user); returns the reply and its latency."""
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    started = time.perf_counter()
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, use_cache=True)
    new_tokens = out[0][inputs["input_ids"].shape[1]:]
    latency_ms = int((time.perf_counter() - started) * 1000)
    return tokenizer.decode(new_tokens, skip_special_tokens=True), latency_ms, len(new_tokens)

In [ ]:
# Sanity check before uploading: the adapter must produce valid extractions.
from datasets import load_dataset

data_dir = snapshot_download(DATASET_REPO, repo_type="dataset", token=HF_TOKEN,
                             allow_patterns=["sft/val.jsonl"])
val = load_dataset("json", data_files=f"{data_dir}/sft/val.jsonl")["train"]
ok = 0
for row in val.select(range(5)):
    text, latency_ms, _ = generate(model, row["messages"][:2])
    try:
        pred = ResolutionExtraction.model_validate_json(text)
        ok += 1
        print("valid", pred.resolution_type.value, f"{latency_ms} ms")
    except Exception as exc:
        print("INVALID:", str(exc)[:200])
assert ok >= 4, "adapter does not look trained; check CHECKPOINT"
SANITY = f"{ok}/5 valid"

In [ ]:
# metrics.json from the trainer state saved with the checkpoint.
state = json.load(open(f"{CHECKPOINT}/trainer_state.json"))
metrics = {
    "base_model": BASE_MODEL,
    "checkpoint": os.path.basename(CHECKPOINT),
    "epochs": state.get("epoch"),
    "global_step": state.get("global_step"),
    "train_loss_log": [h for h in state["log_history"] if "loss" in h],
    "eval_loss_log": [h for h in state["log_history"] if "eval_loss" in h],
    "sanity_check": SANITY,
}
with open(f"{OUTPUT_DIR}/metrics.json", "w") as fh:
    json.dump(metrics, fh, indent=2)
print(json.dumps(metrics["eval_loss_log"], indent=2))

## 8. Save the LoRA adapter (private HF repo)

In [ ]:
model.save_pretrained(f"{OUTPUT_DIR}/final")
tokenizer.save_pretrained(f"{OUTPUT_DIR}/final")
model.push_to_hub(LORA_REPO, token=HF_TOKEN, private=True)
tokenizer.push_to_hub(LORA_REPO, token=HF_TOKEN, private=True)
HfApi(token=HF_TOKEN).upload_file(path_or_fileobj=f"{OUTPUT_DIR}/metrics.json",
                                  path_in_repo="metrics.json", repo_id=LORA_REPO)

## 9. Export a merged Q4_K_M GGUF for CPU serving (llama.cpp)

In [ ]:
model.save_pretrained_gguf("gguf_out", tokenizer, quantization_method="q4_k_m")
candidates = [p for p in glob.glob("**/*.gguf", recursive=True) if "q4_k_m" in p.lower()]
assert candidates, "GGUF export produced no Q4_K_M file"
gguf_path = max(candidates, key=os.path.getmtime)
print(gguf_path, round(os.path.getsize(gguf_path) / 1e9, 2), "GB")

api = HfApi(token=HF_TOKEN)
api.create_repo(GGUF_REPO, private=True, exist_ok=True)
api.upload_file(path_or_fileobj=gguf_path, path_in_repo=GGUF_FILE, repo_id=GGUF_REPO)
print(f"Uploaded https://huggingface.co/{GGUF_REPO}/blob/main/{GGUF_FILE}")

## Next

In the repo's `.env` set `STUDENT_GGUF_REPO=<you>/proxylens-qwen3b-gguf-v1` (and `HF_TOKEN`), then run
the Phase 4 smoke test:

```bash
uv sync --extra student
uv run python -m scripts.smoke_student --n 10
```